# Тестирование DocScanner (Dewarping + Rectification + Background Removal)
Папка: /home/user-4/test_background
Результаты сохраняются там же с суффиксом _result.png

In [1]:
# Шаг 1: Импорты и проверка
import os
import subprocess
import shutil
from PIL import Image
import matplotlib.pyplot as plt
import pytesseract
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

TEST_DIR = "/home/user-4/test_background"
DOCSCANNER_DIR = "/home/user-4/DocScanner"
DISTORTED_DIR = os.path.join(DOCSCANNER_DIR, "distorted")
RECTIFIED_DIR = os.path.join(DOCSCANNER_DIR, "rectified")
SEG_MODEL = os.path.join(DOCSCANNER_DIR, "model_pretrained/seg.pth")
REC_MODEL = os.path.join(DOCSCANNER_DIR, "model_pretrained/DocScanner-L.pth")

# Проверка весов
if not os.path.exists(SEG_MODEL) or not os.path.exists(REC_MODEL):
    print("❌ Ошибка: отсутствуют файлы весов!")
    print("Нужно положить seg.pth и DocScanner-L.pth в model_pretrained/")
    raise FileNotFoundError("Веса не найдены")

print("✅ DocScanner готов к запуску")

✅ DocScanner готов к запуску


In [2]:
# Шаг 2: Функция обработки
def process_docscanner(image_path):
    filename = os.path.basename(image_path)
    base_name = os.path.splitext(filename)[0]
    
    # Очищаем папки
    shutil.rmtree(DISTORTED_DIR, ignore_errors=True)
    shutil.rmtree(RECTIFIED_DIR, ignore_errors=True)
    os.makedirs(DISTORTED_DIR, exist_ok=True)
    os.makedirs(RECTIFIED_DIR, exist_ok=True)
    
    # Копируем изображение в distorted
    shutil.copy(image_path, os.path.join(DISTORTED_DIR, filename))
    
    # Запускаем inference.py с правильными аргументами
    cmd = [
        "python", "inference.py",
        "--seg_model_path", SEG_MODEL,
        "--rec_model_path", REC_MODEL,
        "--distorrted_path", DISTORTED_DIR + "/",
        "--rectified_path", RECTIFIED_DIR + "/"
    ]
    
    result = subprocess.run(cmd, cwd=DOCSCANNER_DIR, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ Ошибка {filename}:\n{result.stderr[:500]}...")
        return None
    
    # Результат в rectified/ имеет суффикс _rec
    rectified_file = os.path.join(RECTIFIED_DIR, f"{base_name}_rec.png")
    if os.path.exists(rectified_file):
        result_path = os.path.join(TEST_DIR, f"{base_name}_result.png")
        shutil.copy(rectified_file, result_path)
        return result_path
    return None

In [3]:
# Шаг 3: Запуск теста
image_files = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

print("🚀 Запуск DocScanner на background...")

for img_file in tqdm(image_files):
    img_path = os.path.join(TEST_DIR, img_file)
    base_name = os.path.splitext(img_file)[0]
    
    original_text = pytesseract.image_to_string(Image.open(img_path), lang='eng+rus', config='--psm 6')
    
    result_path = process_docscanner(img_path)
    
    if result_path:
        processed_text = pytesseract.image_to_string(Image.open(result_path), lang='eng+rus', config='--psm 6')
        
        with open(os.path.join(TEST_DIR, f"{base_name}_comparison.txt"), "w", encoding="utf-8") as f:
            f.write(f"Исходный OCR:\n{original_text}\n\nОбработанный OCR (DocScanner):\n{processed_text}")
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 8))
        axes[0].imshow(Image.open(img_path))
        axes[0].set_title('До (background)')
        axes[0].axis('off')
        axes[1].imshow(Image.open(result_path))
        axes[1].set_title('После DocScanner')
        axes[1].axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(TEST_DIR, f"{base_name}_comparison.png"))
        plt.close()
        
        print(f"✅ {img_file} → {base_name}_result.png")
    else:
        print(f"❌ Не удалось обработать {img_file}")

print("\n🎉 Тестирование завершено! Все _result.png лежат в /home/user-4/test_background")

🚀 Запуск DocScanner на background...


  9%|▉         | 1/11 [00:27<04:35, 27.58s/it]

✅ 0001.jpg → 0001_result.png


 18%|█▊        | 2/11 [00:50<03:41, 24.59s/it]

✅ 0010.jpg → 0010_result.png


...

✅ 0006.jpg → 0006_result.png


 91%|█████████ | 10/11 [04:30<00:28, 28.73s/it]

✅ 0003.jpg → 0003_result.png


100%|██████████| 11/11 [04:50<00:00, 26.44s/it]

✅ 0009.jpg → 0009_result.png

🎉 Тестирование завершено! Все _result.png лежат в /home/user-4/test_background
